# Stage 4 — Model 2: Random Forest (`RandomForestClassifier`)
**Project:** 10-Year Coronary Heart Disease (CHD) Risk Prediction (Framingham Heart Study)  
**Author:** Saif Asaad Naeem | Al-Maaqal University — Department of AI Engineering

---
### Notebook Roadmap:
1. **Environment Setup & Data Loading:** Load `framingham_balanced.csv` and apply the unified 80%/20% stratified split.
2. **Stage 1 — Baseline Model Evaluation:** Evaluate the default `RandomForestClassifier`.
3. **Stage 2 — Continuous Exploration via `RandomizedSearchCV ()`:** Sample hyperparameter distributions using parentheses `()`.
4. **Stage 3 — Exhaustive Exploitation via `GridSearchCV []`:** Refine optimal parameters using an explicit grid `[]`.
5. **Final Results Comparison, Serialization & Diagnostic Plots:** Save `best_model2_randomforest.pkl` and generate diagnostic plots.

### 1. Environment Setup and Data Loading
Load required libraries and configure the 5-Fold Stratified Cross-Validation evaluator.

In [ ]:
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint, uniform

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV, GridSearchCV, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay

df = pd.read_csv('framingham_balanced.csv')
X = df.drop('TenYearCHD', axis=1).values
y = df['TenYearCHD'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

def evaluate_model(model, stage_name):
    cv_res = cross_validate(model, X_train, y_train, cv=skf, scoring=scoring_metrics, n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return {
        'Stage': stage_name,
        'CV_Recall (%)': round(cv_res['test_recall'].mean() * 100, 2),
        'CV_Precision (%)': round(cv_res['test_precision'].mean() * 100, 2),
        'CV_F1 (%)': round(cv_res['test_f1'].mean() * 100, 2),
        'CV_ROC_AUC (%)': round(cv_res['test_roc_auc'].mean() * 100, 2),
        'CV_Accuracy (%)': round(cv_res['test_accuracy'].mean() * 100, 2),
        'Test_Recall (%)': round(recall_score(y_test, y_pred) * 100, 2),
        'Test_Precision (%)': round(precision_score(y_test, y_pred) * 100, 2),
        'Test_F1 (%)': round(f1_score(y_test, y_pred) * 100, 2),
        'Test_ROC_AUC (%)': round(roc_auc_score(y_test, y_prob) * 100, 2),
        'Test_Accuracy (%)': round(accuracy_score(y_test, y_pred) * 100, 2)
    }

print(f"Dataset Loaded: Train = {X_train.shape}, Test = {X_test.shape}")

### 2. Stage 1: Baseline `RandomForestClassifier` Evaluation
Train and evaluate the default `RandomForestClassifier` model.

In [ ]:
base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)
res_base = evaluate_model(base_rf, '1. Baseline RandomForest')
print("Baseline Evaluation:", res_base)

### 3. Stage 2: Broad Exploration via `RandomizedSearchCV ()`
Explore the hyperparameter space using continuous and discrete distributions enclosed in parentheses `()`.

In [ ]:
param_dist_rf = {
    'n_estimators': randint(150, 500),
    'max_depth': randint(10, 32),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5),
    'max_features': uniform(0.3, 0.6)
}

rand_rf = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=25,
    scoring='f1',
    cv=skf,
    random_state=42,
    n_jobs=-1
)
rand_rf.fit(X_train, y_train)
best_r = rand_rf.best_params_
print("Best Parameters from RandomizedSearchCV ():", best_r)
res_rand = evaluate_model(rand_rf.best_estimator_, '2. RandomizedSearchCV ()')

### 4. Stage 3: Fine-Grained Exploitation via `GridSearchCV []`
Refine the optimal hyperparameters using a focused lattice inside square brackets `[]`.

In [ ]:
param_grid_rf = {
    'n_estimators': [max(50, best_r['n_estimators'] - 40), best_r['n_estimators'], best_r['n_estimators'] + 40],
    'max_depth': [max(5, best_r['max_depth'] - 3), best_r['max_depth'], best_r['max_depth'] + 3],
    'min_samples_split': [max(2, best_r['min_samples_split'] - 1), best_r['min_samples_split'], best_r['min_samples_split'] + 1],
    'min_samples_leaf': [max(1, best_r['min_samples_leaf']), best_r['min_samples_leaf'] + 1],
    'max_features': [round(max(0.2, best_r['max_features'] - 0.08), 3), round(best_r['max_features'], 3), round(min(1.0, best_r['max_features'] + 0.08), 3)]
}

grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid=param_grid_rf,
    scoring='f1',
    cv=skf,
    n_jobs=-1
)
grid_rf.fit(X_train, y_train)
print("Best Parameters from GridSearchCV []:", grid_rf.best_params_)
res_grid = evaluate_model(grid_rf.best_estimator_, '3. GridSearchCV [] (Final)')

### 5. Final Results Comparison, Model Serialization & Diagnostic Plots
Print the progression table, save `best_model2_randomforest.pkl`, and plot the Confusion Matrix and ROC curve.

In [ ]:
results_df = pd.DataFrame([res_base, res_rand, res_grid])
print("\n=== Model 2 (RandomForestClassifier) Optimization Progression Table ===")
print(results_df.to_string(index=False))

joblib.dump(grid_rf.best_estimator_, 'best_model2_randomforest.pkl')
print("\nSUMMARY_MODEL_2_RANDOMFOREST:", {'best_params': grid_rf.best_params_, 'metrics': res_grid})

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
y_pred_final = grid_rf.best_estimator_.predict(X_test)
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d', cmap='Greens', ax=axes[0])
axes[0].set_title('RandomForest Confusion Matrix (Test Set)', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
RocCurveDisplay.from_estimator(grid_rf.best_estimator_, X_test, y_test, ax=axes[1], name='RandomForest (GridSearch)')
axes[1].set_title('RandomForest ROC Curve', fontweight='bold')
plt.tight_layout()
plt.savefig('fig_model2_randomforest.png', dpi=300)
plt.show()